# Publication extension — MD Mahmudul Hasan
## What this notebook does
Tests whether local-only versus source-based training conclusions are stable across repeated data partitions, source/target overlap policies, and explicitly budgeted local model selection.

**This is a new protocol. Its results must not silently replace the old thesis results.**
It focuses on XGBoost, not all thesis models. No new empirical results exist until you run it.
The closest preprint already studies source benefit: https://doi.org/10.20944/preprints202610.1041.v1 . These checks improve evidence but do not by themselves guarantee novelty or publication.

## Run instructions
1. Use a fresh Google Colab runtime. Put your same four original CSV datasets in `MyDrive/thesis_data`.
2. Run the installation cell. If Colab asks for a runtime restart, restart and continue from the imports cell.
3. Run once with `QUICK_MODE = True`. It uses smaller samples, one split, one seed and one budget. It still reads all CSV chunks to sample uniformly, so startup may be slow.
4. If the quick run succeeds, change to `QUICK_MODE = False` and run all cells. Full mode uses five partition seeds, two training seeds, three label budgets and two overlap policies.
5. The final cell writes a ZIP to Drive. Send that ZIP back for analysis. Do not send the raw 16 GB data.

Keep the default configuration for the first full run. Quick outputs are pipeline checks, not publication evidence. Checkpoints resume finished blocks after a disconnection. Data are re-read on restart; source models and completed blocks are reused only when code, data, versions and settings match.


In [ ]:
%pip install -q xgboost==3.1.2


In [ ]:
import os, json, glob, hashlib, platform, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.model_selection import train_test_split
from IPython.display import display
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')


In [ ]:
QUICK_MODE = True
DATA_DIR = '/content/drive/MyDrive/thesis_data'
RESULTS_ROOT = '/content/drive/MyDrive/thesis_publication_extension'
DATASETS = {
    'ToN-IoT': 'NF-ToN-IoT-v3*.csv',
    'BoT-IoT': 'NF-BoT-IoT-v3*.csv',
    'UNSW-NB15': 'NF-UNSW-NB15-v3*.csv',
    'CIC-IDS2018': '*CIC*2018*.csv',
}
IOT_DATASETS = ['ToN-IoT','BoT-IoT']
CAP_BENIGN = 5000 if QUICK_MODE else 150000
CAP_PER_ATTACK = 1500 if QUICK_MODE else 30000
CHUNK_SIZE = 200000
CFG = {
    'quick': QUICK_MODE,
    'targets': list(DATASETS),
    'split_seeds': [123] if QUICK_MODE else [123, 456, 789, 1011, 1213],
    'model_seeds': [0] if QUICK_MODE else [0, 1],
    'fractions': [0.10] if QUICK_MODE else [0.01, 0.05, 0.10],
    'overlap_policies': ['retain_overlap','exclude_target_overlap'],
    'source_cap': 15000 if QUICK_MODE else 300000,
    'trees': 20 if QUICK_MODE else 200,
    'validation_fpr_target': 0.01,
    'sampling_seed': 42, 'benign_cap': CAP_BENIGN, 'attack_cap': CAP_PER_ATTACK,
    'candidates': [
        {'max_depth': 3,'learning_rate': 0.05,'subsample': 0.8,'colsample_bytree': 0.8},
        {'max_depth': 6,'learning_rate': 0.05,'subsample': 0.8,'colsample_bytree': 0.8},
        {'max_depth': 3,'learning_rate': 0.10,'subsample': 0.8,'colsample_bytree': 0.8},
    ],
}
if QUICK_MODE: CFG['candidates'] = CFG['candidates'][:1]
assert os.path.isdir(DATA_DIR), f'Missing folder {DATA_DIR}'
print('QUICK TEST' if QUICK_MODE else 'FULL EXPERIMENT')
blocks = len(CFG['targets'])*len(CFG['split_seeds'])*len(CFG['model_seeds'])*len(CFG['fractions'])*len(CFG['overlap_policies'])*3
print('Completed experiment blocks expected:', blocks)
print('Full mode runs 720 blocks, each with 3 candidates, plus source pretraining. Runtime depends on Colab.')


## Input checks and provenance
File hashes identify the actual data release. Only CSV paths matching the listed patterns are read. Do not put duplicate copies of a dataset in this directory. SHA-256 hashing can take several minutes over Drive.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
inputs=[]; used=set(); seen_hashes=set()
for dataset,pattern in DATASETS.items():
    paths=sorted(glob.glob(f'{DATA_DIR}/**/{pattern}',recursive=True))
    assert paths, f'Missing {dataset}: {pattern}'
    for path in paths:
        real=os.path.realpath(path)
        assert real not in used, 'A file matched two dataset patterns'
        used.add(real)
        print('Fingerprinting',dataset,os.path.basename(path),flush=True)
        h=sha256_file(path)
        assert h not in seen_hashes, 'Duplicate CSV content found; remove duplicate copy from input folder'
        seen_hashes.add(h)
        inputs.append({'dataset':dataset,'relative_path':os.path.relpath(path,DATA_DIR),'bytes':os.path.getsize(path),'sha256':h})
CFG['inputs']=inputs
PREP_DIR=Path(RESULTS_ROOT)/'preprocessing'/hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
PREP_DIR.mkdir(parents=True,exist_ok=True)
def save_table(df,name): df.to_csv(PREP_DIR/(name+'.csv'),index=False)


## Load and clean the data
Preserves the earlier notebook's per-original-label sampling caps and label mapping. These caps change class prevalence and are a benchmark design choice. Sampling seed stays fixed; later split repeats do not measure raw-sampling uncertainty.
Identifiers are removed. Conflicting duplicate vectors are removed within each dataset. Exact matching uses the 46-feature representation, not raw packet identity. Cross-dataset matching is a sensitivity audit and does not identify every kind of near-duplicate or capture leakage.


In [ ]:
LABEL_COL, ATTACK_COL = 'Label', 'Attack'
def load_sampled(pattern, seed=42):
    files=sorted(glob.glob(f'{DATA_DIR}/**/{pattern}',recursive=True))
    assert files
    rng=np.random.default_rng(seed); kept={}; counts_total={}
    for f in files:
        offset=0
        for chunk in pd.read_csv(f,chunksize=CHUNK_SIZE,low_memory=False):
            chunk[ATTACK_COL]=chunk[ATTACK_COL].astype(str).str.strip()
            chunk['_priority']=rng.random(len(chunk))
            chunk['_row_id']=[f'{os.path.relpath(f,DATA_DIR)}:{j}' for j in range(offset,offset+len(chunk))]
            offset+=len(chunk)
            for cls,g in chunk.groupby(ATTACK_COL):
                counts_total[cls]=counts_total.get(cls,0)+len(g)
                cap=CAP_BENIGN if cls.lower()=='benign' else CAP_PER_ATTACK
                prev=kept.get(cls)
                combined=pd.concat([prev,g],ignore_index=True) if prev is not None else g
                kept[cls]=combined.nsmallest(cap,'_priority')
    return pd.concat(kept.values(),ignore_index=True).drop(columns='_priority'),counts_total
raw,full_counts={},{}
for name,pat in DATASETS.items():
    raw[name],full_counts[name]=load_sampled(pat)
    print(name,len(raw[name]),'sampled rows')


In [ ]:
IDENTIFIERS = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT',
               'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'DNS_QUERY_ID']

LABEL_MAP = {
 'benign':'Benign','injection':'Web/Injection','xss':'Web/Injection',
 'password':'Brute force','dos':'DoS/DDoS','ddos':'DoS/DDoS',
 'backdoor':'Botnet/Malware','mitm':'Other','scanning':'Reconnaissance',
 'ransomware':'Botnet/Malware','reconnaissance':'Reconnaissance','theft':'Other',
 'exploits':'Other','fuzzers':'Other','generic':'Other','shellcode':'Botnet/Malware',
 'analysis':'Other','worms':'Botnet/Malware','dos_attacks-hulk':'DoS/DDoS',
 'ddos_attacks-loic-http':'DoS/DDoS','ssh-bruteforce':'Brute force',
 'dos_attacks-goldeneye':'DoS/DDoS','ddos_attack-hoic':'DoS/DDoS',
 'infilteration':'Botnet/Malware','infiltration':'Botnet/Malware',
 'dos_attacks-slowloris':'DoS/DDoS','ddos_attack-loic-udp':'DoS/DDoS',
 'bot':'Botnet/Malware','brute_force_-web':'Brute force',
 'sql_injection':'Web/Injection','brute_force_-xss':'Web/Injection',
 'dos_attacks-slowhttptest':'DoS/DDoS','ftp-bruteforce':'Brute force'}
def harmonize_label(a):
    key=str(a).strip().lower()
    if key not in LABEL_MAP: raise ValueError(f'Unknown attack label: {a!r}; review mapping before continuing.')
    return LABEL_MAP[key]

common = set.intersection(*[set(df.columns) for df in raw.values()])
FEATURES = sorted(c for c in common if c not in IDENTIFIERS + [LABEL_COL, ATTACK_COL, '_row_id'])
print(f'{len(FEATURES)} shared features used:\n', FEATURES)

data, stats_rows, map_rows = {}, [], []
for name, df in raw.items():
    n0 = len(df)
    df = df[FEATURES + [LABEL_COL, ATTACK_COL, '_row_id']].copy()
    df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    n1 = len(df)
    labels = pd.to_numeric(df[LABEL_COL],errors='raise')
    assert set(labels.unique()) <= {0,1}, 'Unexpected binary label values'
    df['y'] = labels.astype(int)
    df['cls'] = df[ATTACK_COL].map(harmonize_label)
    hashes=pd.util.hash_pandas_object(df[FEATURES],index=False)
    df['_feature_hash']=hashes.astype(str)
    conflict=df.groupby('_feature_hash').y.transform('nunique').gt(1)
    ambiguous=df.groupby('_feature_hash').cls.transform('nunique').gt(1)
    df.loc[ambiguous,'cls']='Ambiguous'
    df=df[~conflict].sort_values('_row_id').drop_duplicates(subset=FEATURES).copy()
    assert not df.duplicated(subset=FEATURES).any()
    df['dataset'] = name
    data[name] = df.reset_index(drop=True)
    for a, c in df[ATTACK_COL].value_counts().items():
        map_rows.append({'Dataset': name, 'Original label': a, 'Harmonized class': harmonize_label(a),
                         'Flows in full dataset': full_counts[name].get(a, ''), 'Flows after sampling and cleaning': c})
    stats_rows.append({'Dataset': name, 'IoT': name in IOT_DATASETS,
                       'Flows in full dataset': sum(full_counts[name].values()),
                       'Sampled': n0, 'After removing invalid': n1, 'After deduplication': len(df),
                       'Benign': int((df.y == 0).sum()), 'Attack': int((df.y == 1).sum()),
                       'Attack %': round(100 * df.y.mean(), 2), 'Features': len(FEATURES)})

stats = pd.DataFrame(stats_rows); save_table(stats, 'table4_2_dataset_statistics')
mapping = pd.DataFrame(map_rows); save_table(mapping, 'table3_2_label_mapping')
pd.set_option('display.max_rows', 200)
display(stats); display(mapping)
# Save sample provenance and exact-feature overlap counts, without raw network identifiers.
for name,df in data.items():
    save_table(df[['_row_id','_feature_hash','y','cls']],f'sample_manifest_{name}')
overlap=[]
for a in data:
    for b in data:
        if a!=b: overlap.append({'Source':a,'Target':b,'Shared feature hashes':len(set(data[a]._feature_hash)&set(data[b]._feature_hash))})
save_table(pd.DataFrame(overlap),'cross_dataset_overlap_audit')
assert len(FEATURES)==46, f'Unexpected schema: {len(FEATURES)} features. Inspect before proceeding.'

del raw
import gc
gc.collect()


## Protocol fixed before examining these results
- One 50/50 stratified pool/test split per dataset and partition seed. Source data use the other datasets' pool halves; no source validation subset is required by this extension.
- Random local label acquisition: a seeded permutation of the target pool, without using labels to choose examples. The 1%, 5%, 10% budgets are nested and refer to the cleaned target dataset.
- Each acquired budget is split 70/30 for training versus selection/calibration. **Both parts count toward the budget.** Hyperparameters and thresholds use the validation part; no final refit occurs after calibration.
- The same candidate grid is tested for each method, selected by validation macro F1 at threshold 0.5. Target-only and pooled models each have 200 trees in full mode; continuation has 100 source trees plus 100 local trees. This matches tree count and search opportunities, not compute cost or optimality.
- Source pooling removes internally conflicting vectors and deduplicates source features before capping rows. Compare retaining versus removing source vectors whose features match any target vector. Removal uses target features, including the test partition, only for this explicitly transductive benchmark audit; no target test labels are used. It is not an inductive production procedure.
- Primary endpoint: held-out macro F1 at 0.5. Report fine-tuned minus local-only and pooled minus local-only per target, budget, split and seed. Average model seeds within a split first. Do not treat seeds, splits or overlap scenarios as independent networks.
- Secondary endpoint: recall and achieved test FPR at a validation-selected threshold whose empirical validation FPR is at most 1%. This is not a guarantee of 1% FPR on future traffic. Calibration with fewer than 100 benign validation samples is flagged as sparse. Model selection and calibration reuse the validation subset, so validation performance is not an unbiased estimate.
- Class weights are recomputed from each fitting set. Per-source weighting and alternative tuning grids are not tested. Source-only and other thesis models remain in the original thesis benchmark, not this extension.


In [ ]:
CORE_SOURCE = '"""Publication extension v1. Research protocol, not a deployment evaluation."""\nimport os, json, hashlib, time, platform, importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import (f1_score, confusion_matrix, average_precision_score,\n                             balanced_accuracy_score, roc_curve)\n\nCODE_VERSION = \'publication-extension-1.0\'\n\ndef digest(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()\n\ndef atomic_json(path, obj):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))\n    os.replace(tmp, path)\n\ndef metrics(y, scores, threshold=0.5):\n    pred = np.asarray(scores) >= threshold\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return dict(macro_f1=float(f1_score(y, pred, average=\'macro\', labels=[0,1], zero_division=0)),\n                balanced_accuracy=float(balanced_accuracy_score(y, pred)),\n                average_precision=float(average_precision_score(y, scores)),\n                recall=float(tp / max(tp + fn, 1)), fpr=float(fp / max(fp + tn, 1)),\n                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))\n\ndef select_threshold(y, scores, max_fpr=0.01):\n    """Select on validation only. >= tie rule matches metrics. No test access."""\n    fpr, tpr, thresholds = roc_curve(y, scores, drop_intermediate=False)\n    feasible = np.flatnonzero(fpr <= max_fpr)\n    # Highest recall, then lowest FPR, then highest threshold, deterministically.\n    i = max(feasible, key=lambda k: (tpr[k], -fpr[k], thresholds[k]))\n    t = float(thresholds[i])\n    # A finite threshold above all probability scores means no alerts.\n    return 1.000001 if not np.isfinite(t) else t\n\ndef prepare_data(data, features):\n    out = {}\n    for name, frame in data.items():\n        d = frame.copy().reset_index(drop=True)\n        d[features] = d[features].astype(\'float64\')\n        assert np.isfinite(d[features].to_numpy()).all()\n        assert set(d.y.unique()) == {0,1}\n        assert not d.duplicated(subset=features).any()\n        # Canonical dtype and signed zero for matching across files.\n        d[features] = d[features].mask(d[features] == 0, 0.0)\n        d[\'_feature_hash\'] = pd.util.hash_pandas_object(d[features], index=False).astype(str)\n        assert d._feature_hash.nunique() == len(d), \'Hash collision or duplicate: inspect before running.\'\n        out[name] = d\n    return out\n\ndef split_indices(data, seed):\n    splits = {}\n    for name,d in data.items():\n        pool,test = train_test_split(np.arange(len(d)), test_size=.5, stratify=d.y,\n                                     random_state=seed)\n        assert not set(pool) & set(test)\n        splits[name] = (pool,test)\n    return splits\n\ndef budget_indices(y, pool, total_n, fractions, seed):\n    """Random acquisition without consulting labels; nested budgets."""\n    order = np.random.default_rng(seed).permutation(pool)\n    result = {}\n    for f in sorted(fractions):\n        n = max(1, int(np.floor(total_n*f)))\n        if n > len(pool): raise ValueError(\'Budget exceeds non-test pool\')\n        result[f] = order[:n]\n    return result\n\ndef make_source(data, features, splits, target, policy, cap, seed):\n    src = pd.concat([d.iloc[splits[n][0]] for n,d in data.items() if n != target], ignore_index=True)\n    original_n = len(src)\n    # Standard policy for duplicate source vectors and inconsistent source labels.\n    conflict = src.groupby(\'_feature_hash\').y.transform(\'nunique\').gt(1)\n    conflict_n = int(conflict.sum())\n    src = src[~conflict].drop_duplicates(subset=features).copy()\n    after_source_cleanup = len(src)\n    target_hashes = set(data[target]._feature_hash)\n    overlap = src._feature_hash.isin(target_hashes)\n    matched_n = int(overlap.sum())\n    if policy == \'exclude_target_overlap\': src = src[~overlap].copy()\n    elif policy != \'retain_overlap\': raise ValueError(policy)\n    if len(src) > cap:\n        idx,_ = train_test_split(np.arange(len(src)), train_size=cap, stratify=src.y, random_state=seed)\n        src = src.iloc[idx]\n    assert set(src.y.unique()) == {0,1}\n    if policy == \'exclude_target_overlap\': assert not set(src._feature_hash) & target_hashes\n    return src, dict(source_before=original_n, source_conflicting_rows=conflict_n,\n                    source_after_cleanup=after_source_cleanup, source_matches_target=matched_n,\n                    source_used=len(src), source_matches_target_after_cap=int(src._feature_hash.isin(target_hashes).sum()))\n\ndef matrix(d, features):\n    v = d[features].to_numpy(dtype=np.float64)\n    return (np.sign(v)*np.log1p(np.abs(v))).astype(np.float32)\n\ndef fit_model(X, y, candidate, trees, seed, initial=None):\n    m = xgb.XGBClassifier(**candidate, n_estimators=trees, tree_method=\'hist\',\n                          objective=\'binary:logistic\', eval_metric=\'logloss\',\n                          random_state=seed, n_jobs=2,\n                          scale_pos_weight=float(np.sum(y==0)/max(np.sum(y==1),1)))\n    m.fit(X, y, xgb_model=initial)\n    expected = trees + (initial.num_boosted_rounds() if initial is not None else 0)\n    assert m.get_booster().num_boosted_rounds() == expected\n    return m\n\ndef run_experiments(data, features, cfg, root, max_blocks=None):\n    """Resume-safe unit = target/split/model seed/budget/overlap policy/method."""\n    root=Path(root); data=prepare_data(data, features)\n    signatures={n: digest(d[[\'_feature_hash\',\'y\',\'_row_id\']].to_dict(\'list\')) for n,d in data.items()}\n    versions={n:importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'scikit-learn\',\'xgboost\']}\n    provenance=dict(code_version=CODE_VERSION, core_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                    config=cfg, features=features, data_signatures=signatures,\n                    versions=versions, python=platform.python_version())\n    run_id=digest(provenance)[:16]; out=root / (\'quick_\' if cfg[\'quick\'] else \'full_\') / run_id\n    for sub in [\'checkpoints\',\'predictions\',\'manifests\',\'source_models\',\'tables\',\'figures\']:\n        (out/sub).mkdir(parents=True,exist_ok=True)\n    atomic_json(out/\'config.json\',provenance)\n    count=0\n    for split_seed in cfg[\'split_seeds\']:\n        splits=split_indices(data,split_seed)\n        for target in cfg[\'targets\']:\n            d=data[target]; pool,test=splits[target]\n            manifest=d[[\'_row_id\',\'_feature_hash\',\'y\',\'cls\']].copy()\n            manifest[\'partition\']=\'pool\';manifest.loc[test,\'partition\']=\'test\'\n            manifest.to_csv(out/\'manifests\'/f\'{target}_split{split_seed}.csv.gz\',index=False)\n            Xtest=matrix(d.iloc[test],features); ytest=d.iloc[test].y.to_numpy()\n            budgets=budget_indices(d.y,pool,len(d),cfg[\'fractions\'],split_seed+10000)\n            # Acquisition and inner split are independent of the model seed and policy.\n            for fraction, acquired in budgets.items():\n                vals,counts=np.unique(d.iloc[acquired].y,return_counts=True)\n                if len(vals)<2 or min(counts)<4:\n                    raise ValueError(f\'{target}/{fraction}: too few acquired examples per class. Do not secretly resample; revise protocol explicitly.\')\n                train,val=train_test_split(acquired,test_size=.3,stratify=d.iloc[acquired].y,random_state=split_seed+20000)\n                assert set(train).isdisjoint(val) and set(acquired).isdisjoint(test)\n                label_manifest=d.iloc[acquired][[\'_row_id\',\'_feature_hash\',\'y\',\'cls\']].copy()\n                label_manifest[\'role\']=np.where(label_manifest.index.isin(train),\'train\',\'selection_and_calibration\')\n                label_manifest.to_csv(out/\'manifests\'/f\'{target}_split{split_seed}_budget{fraction}.csv.gz\',index=False)\n                Xt=matrix(d.iloc[train],features); yt=d.iloc[train].y.to_numpy()\n                Xv=matrix(d.iloc[val],features); yv=d.iloc[val].y.to_numpy()\n                for model_seed in cfg[\'model_seeds\']:\n                    for policy in cfg[\'overlap_policies\']:\n                        src,audit=make_source(data,features,splits,target,policy,cfg[\'source_cap\'],split_seed)\n                        Xs=matrix(src,features); ys=src.y.to_numpy()\n                        for method in [\'target_only\',\'fine_tune\',\'pooled\']:\n                            key=dict(target=target,split_seed=split_seed,model_seed=model_seed,\n                                     fraction=fraction,overlap_policy=policy,method=method)\n                            token=digest(key)[:24];checkpoint=out/\'checkpoints\'/f\'{token}.json\'\n                            if checkpoint.exists():\n                                saved=json.loads(checkpoint.read_text());assert saved[\'key\']==key and saved[\'run_id\']==run_id\n                                continue\n                            print(\'Running\',key,flush=True)\n                            start=time.perf_counter(); candidates=[]; best=None; best_f1=-1.0\n                            source_fit_seconds=0.0\n                            for ci,candidate in enumerate(cfg[\'candidates\']):\n                                initial=None\n                                if method==\'fine_tune\':\n                                    skey=dict(target=target,split_seed=split_seed,model_seed=model_seed,policy=policy,candidate=ci)\n                                    spath=out/\'source_models\'/f\'{digest(skey)[:24]}.ubj\'\n                                    stime=spath.with_suffix(\'.json\')\n                                    if not (spath.exists() and stime.exists()):\n                                        t=time.perf_counter()\n                                        source_model=fit_model(Xs,ys,candidate,cfg[\'trees\']//2,model_seed)\n                                        source_model.get_booster().save_model(spath)\n                                        atomic_json(stime,{\'fit_seconds\':time.perf_counter()-t})\n                                    source_fit_seconds+=json.loads(stime.read_text())[\'fit_seconds\']\n                                    initial=xgb.Booster();initial.load_model(spath)\n                                if method==\'pooled\': Xfit=np.concatenate([Xs,Xt]);yfit=np.concatenate([ys,yt])\n                                else: Xfit,yfit=Xt,yt\n                                ntrees=cfg[\'trees\']//2 if method==\'fine_tune\' else cfg[\'trees\']\n                                m=fit_model(Xfit,yfit,candidate,ntrees,model_seed,initial)\n                                pval=m.predict_proba(Xv)[:,1]\n                                score=metrics(yv,pval)[\'macro_f1\']\n                                candidates.append(dict(index=ci,params=candidate,validation_macro_f1=score))\n                                if score>best_f1:\n                                    best_f1=score;best=m;best_ci=ci;best_val=pval\n                            # No retraining after threshold calibration; test is touched only now.\n                            threshold=select_threshold(yv,best_val,cfg[\'validation_fpr_target\'])\n                            t=time.perf_counter(); ptest=best.predict_proba(Xtest)[:,1]\n                            prediction_seconds=time.perf_counter()-t\n                            elapsed=time.perf_counter()-start\n                            common={**key,**audit,\'acquired_labels\':len(acquired),\'fit_labels\':len(train),\n                                    \'selection_labels\':len(val),\'validation_benign\':int(sum(yv==0)),\n                                    \'validation_attack\':int(sum(yv==1)),\'test_n\':len(test),\n                                    \'best_candidate\':best_ci,\'validation_macro_f1\':best_f1,\n                                    \'elapsed_seconds\':elapsed,\'source_pretraining_seconds_all_candidates\':source_fit_seconds,\n                                    \'prediction_seconds\':prediction_seconds,\'total_trees\':best.get_booster().num_boosted_rounds(),\n                                    \'low_fpr_calibration_sparse\':bool(sum(yv==0)<100)}\n                            rows=[]\n                            for setting,thr in [(\'fixed_0.5\',0.5),(\'validation_fpr_1pct\',threshold)]:\n                                rows.append({**common,\'threshold_setting\':setting,\'threshold\':thr,\n                                             \'validation_fpr\':metrics(yv,best_val,thr)[\'fpr\'],\n                                             **metrics(ytest,ptest,thr)})\n                            predpath=out/\'predictions\'/f\'{token}.npz\'\n                            np.savez_compressed(predpath,y=ytest,score=ptest,\n                                                feature_hash=d.iloc[test]._feature_hash.to_numpy(dtype=str),\n                                                attack_family=d.iloc[test].cls.to_numpy(dtype=str))\n                            atomic_json(checkpoint,dict(run_id=run_id,key=key,rows=rows,candidates=candidates))\n                            count+=1\n                            if max_blocks is not None and count>=max_blocks:return out\n    return out\n\ndef summarize(out):\n    """Descriptive paired effects. Repeated partitions are not independent networks."""\n    out=Path(out); rows=[]\n    for file in sorted((out/\'checkpoints\').glob(\'*.json\')): rows.extend(json.loads(file.read_text())[\'rows\'])\n    if not rows:raise ValueError(\'No completed runs\')\n    r=pd.DataFrame(rows);r.to_csv(out/\'tables\'/\'all_runs.csv\',index=False)\n    cfg=json.loads((out/\'config.json\').read_text())[\'config\']\n    expected=len(cfg[\'targets\'])*len(cfg[\'split_seeds\'])*len(cfg[\'model_seeds\'])*len(cfg[\'fractions\'])*len(cfg[\'overlap_policies\'])*3\n    complete=len(rows)==expected*2\n    atomic_json(out/\'completion.json\',dict(complete=complete,completed_blocks=len(rows)//2,expected_blocks=expected))\n    keys=[\'target\',\'split_seed\',\'model_seed\',\'fraction\',\'overlap_policy\',\'threshold_setting\']\n    control=r[r.method==\'target_only\'][keys+[\'macro_f1\',\'recall\',\'fpr\']].rename(columns={x:\'local_\'+x for x in [\'macro_f1\',\'recall\',\'fpr\']})\n    paired=r[r.method!=\'target_only\'].merge(control,on=keys,validate=\'many_to_one\')\n    for metric in [\'macro_f1\',\'recall\',\'fpr\']:paired[\'delta_\'+metric]=paired[metric]-paired[\'local_\'+metric]\n    paired.to_csv(out/\'tables\'/\'paired_effects.csv\',index=False)\n    # Average model seeds within each split before descriptive split summaries.\n    g=[\'target\',\'fraction\',\'overlap_policy\',\'threshold_setting\',\'method\']\n    sm=paired.groupby(g+[\'split_seed\'])[[\'delta_macro_f1\',\'delta_recall\',\'delta_fpr\']].mean().reset_index()\n    sm.to_csv(out/\'tables\'/\'split_effects.csv\',index=False)\n    stats=sm.groupby(g).agg(mean_delta_f1=(\'delta_macro_f1\',\'mean\'),\n                            min_delta_f1=(\'delta_macro_f1\',\'min\'),max_delta_f1=(\'delta_macro_f1\',\'max\'),\n                            sd_across_splits=(\'delta_macro_f1\',\'std\'),n_splits=(\'split_seed\',\'nunique\'),\n                            positive_split_fraction=(\'delta_macro_f1\',lambda s:float((s>0).mean()))).reset_index()\n    stats.to_csv(out/\'tables\'/\'target_effect_summary.csv\',index=False)\n    # Targets contribute equally; no p-values or network-population CI from four targets.\n    stats.groupby([\'fraction\',\'overlap_policy\',\'threshold_setting\',\'method\']).mean(numeric_only=True).to_csv(out/\'tables\'/\'equal_target_descriptive_summary.csv\')\n    import matplotlib.pyplot as plt\n    for policy in cfg[\'overlap_policies\']:\n        fig,axs=plt.subplots(1,len(cfg[\'targets\']),figsize=(4*len(cfg[\'targets\']),3.6),squeeze=False)\n        for ax,target in zip(axs[0],cfg[\'targets\']):\n            sub=stats[(stats.target==target)&(stats.overlap_policy==policy)&(stats.threshold_setting==\'fixed_0.5\')]\n            for method,z in sub.groupby(\'method\'):\n                z=z.sort_values(\'fraction\');x=100*z.fraction.to_numpy();y=z.mean_delta_f1.to_numpy()\n                ax.plot(x,y,marker=\'o\',label=method)\n                ax.fill_between(x,z.min_delta_f1.to_numpy(),z.max_delta_f1.to_numpy(),alpha=.12)\n            ax.axhline(0,color=\'black\',lw=.7);ax.set_title(target);ax.set_xlabel(\'Total acquired labels (%)\');ax.set_ylabel(\'Macro F1 minus local-only\');ax.legend(fontsize=7)\n        fig.suptitle(\'Descriptive split range (not a confidence interval)\' + (\'\' if complete else \' — INCOMPLETE RUN\'))\n        fig.tight_layout();fig.savefig(out/\'figures\'/f\'local_advantage_{policy}.png\',dpi=200);plt.close(fig)\n    return r,stats\n'
Path("publication_core.py").write_text(CORE_SOURCE)
import importlib.util
spec=importlib.util.spec_from_file_location("publication_core", "publication_core.py")
core=importlib.util.module_from_spec(spec)
spec.loader.exec_module(core)


In [ ]:
OUT=core.run_experiments(data,FEATURES,CFG,RESULTS_ROOT)
print('Experiment outputs:',OUT)


## Summaries and export
A complete full run contains 720 checkpoint blocks and 1,440 metric rows (two threshold settings per block). Fixed-threshold comparisons and calibrated-threshold comparisons use the same trained models and are not independent experiments.
Plots show descriptive minima/maxima across repeated splits, not confidence intervals. Four target testbeds are insufficient for a general population claim. No automatic significance claims are produced. Source pretraining timing and incremental timing are separately logged; cached versus uncached wall times cannot be compared as equal end-to-end cost.


In [ ]:
runs,summary=core.summarize(OUT)
display(summary)
completion=json.loads((OUT/'completion.json').read_text())
print(completion)
assert completion['complete'], 'Run all experiment blocks before exporting final evidence.'
for f in PREP_DIR.glob('*.csv'): shutil.copy2(f,OUT/'tables'/f.name)
(OUT/'requirements_observed.txt').write_text(subprocess.check_output(['python','-m','pip','freeze'],text=True))
(OUT/'experiment_core.py').write_text(CORE_SOURCE)
archive=shutil.make_archive(str(OUT.parent/('publication_results_'+OUT.name)), 'zip',root_dir=OUT)
print('Send this ZIP back:',archive)
print('Quick mode outputs are testing only.' if QUICK_MODE else 'Full computation complete; results still need scientific review.')
